# Chapter 2: count the batches three ways

Section 2.6 asks for the six brew week, counted by brew, by fill and by date code. Kestrel's rule (D03 on the
decision log) lets a later brew join a fill only within 24 hours of the first. Change the window and watch the
count move. All data is synthetic.

In [ ]:
import sys
sys.path.insert(0, "..")  # the companion folder, where kestrel.py lives
import pandas as pd
import kestrel as k

brew = k.load("brew")
batch = k.load("batch")
bridge = k.load("brew_batch")
movement = k.load("movement")
lot = k.load("lot")

Knockout is when the wort leaves the brewhouse for the fermenter. The model holds it as the time of the
produce movement from the brew to the fill.

In [ ]:
produce = movement[movement.movement_type == "produce"][["from_node_key", "to_node_key", "posted_utc"]]
week = (brew.drop(columns="vessel_key").merge(produce, left_on="brew_key", right_on="from_node_key")
            .merge(batch[["batch_key", "batch_id", "vessel_key"]], left_on="to_node_key", right_on="batch_key"))
week["knockout_utc"] = pd.to_datetime(week.posted_utc)
week = week.sort_values("knockout_utc")
print(week[["brew_id", "batch_id", "knockout_utc"]].to_string(index=False))

Date codes: walk each fill forward to its bright tank fill, then to the pack lots printed on the cans and bottles.

In [ ]:
lots = lot.set_index("lot_key")
to_bright = movement[(movement.movement_type == "transfer") & (movement.from_node_type == "batch")]
to_pack = movement[(movement.movement_type == "pack") & (movement.to_node_type == "pack_lot")]
codes = (to_bright[["from_node_key", "to_node_key"]].rename(columns={"from_node_key": "batch_key", "to_node_key": "bright"})
         .merge(to_pack[["from_node_key", "to_node_key"]], left_on="bright", right_on="from_node_key"))
codes["pack_lot"] = codes.to_node_key.map(lots.lot_id)
in_week = codes[codes.batch_key.isin(week.batch_key)]

print("Batches by brew:     ", week.brew_id.nunique())
print("Batches by fill:     ", week.batch_id.nunique())
print("Batches by date code:", in_week.pack_lot.nunique(), sorted(in_week.pack_lot))

Now apply the rule yourself. A brew joins the open fill in its fermenter only if its knockout is within the
window of the first brew in that fill. Otherwise it starts a new batch.

In [ ]:
def count_fills(week, window_hours):
    batches, open_fill = 0, {}
    for _, b in week.iterrows():
        first = open_fill.get(b.vessel_key)
        if first is None or (b.knockout_utc - first).total_seconds() / 3600 > window_hours:
            batches += 1
            open_fill[b.vessel_key] = b.knockout_utc
    return batches

for hours in (24, 12):
    print(f"window {hours} hours: {count_fills(week, hours)} batches")

At 12 hours the second pale ale brew (knockout 18 hours 50 minutes after the first) starts its own batch,
so the count moves from 3 to 4. The rule is a decision for the head brewer, written on the log.